# Scgraph hot fixer

## Find nodes and its edges

- Enter (lat lon)

In [9]:
import math

# 載入 scgraph 預設海運圖資
my_geograph = None
try:
    from scgraph.geographs.marnet import marnet_geograph as my_geograph  # type: ignore
except Exception:
    print("圖資載入失敗")

# 您想尋找的目標地點座標 (請替換成您想封路的實際經緯度)
target_lat = 14.408356  
target_lon = 119.868552
# 14.408356, 119.868552

if my_geograph is not None:
    # 1. 找出距離這個座標最近的節點 ID
    closest_node_id = None
    min_distance = float('inf')

    # 走訪所有的節點來找最近的點
    for node_id, coords in enumerate(my_geograph.nodes):
        # 修正重點：因為 coords 是一個列表 [latitude, longitude]，所以改用索引 0 和 1 取值
        dist = math.hypot(coords[0] - target_lat, coords[1] - target_lon)
        if dist < min_distance:
            min_distance = dist
            closest_node_id = node_id

    print(f" 找到最近的節點 ID: {closest_node_id}")
    print(f" 該節點的座標 [緯度, 經度]: {my_geograph.nodes[closest_node_id]}\n")

    # 2. 查看這個節點連接著哪些邊 (Edges)
    # my_geograph.graph 是一個列表，裡面存放每個節點相連的 {目標節點: 權重} 字典
    connected_edges = my_geograph.graph[closest_node_id]

    print(f" 節點 {closest_node_id} 連接的邊有:")
    for target_node, weight in connected_edges.items():
        print(f" -> 通往節點 ID: {target_node} | 原始權重(距離): {weight}")

 找到最近的節點 ID: 6872
 該節點的座標 [緯度, 經度]: [14.4, 119.9]

 節點 6872 連接的邊有:
 -> 通往節點 ID: 4562 | 原始權重(距離): 360.201
 -> 通往節點 ID: 1533 | 原始權重(距離): 536.71
 -> 通往節點 ID: 7386 | 原始權重(距離): 567.5
 -> 通往節點 ID: 6754 | 原始權重(距離): 9999999
 -> 通往節點 ID: 4512 | 原始權重(距離): 284.107
 -> 通往節點 ID: 10544 | 原始權重(距離): 294.018


## plot the node and the connections

In [10]:
import folium
import webbrowser
import os

# 確保這是在您已經載入 my_geograph 且找到 closest_node_id (6872) 的環境下執行

center_node_id = closest_node_id # 也就是 6872
center_coords = my_geograph.nodes[center_node_id] # [14.4, 119.9]

# 1. 建立 Folium 地圖，中心點設為您的目標節點，適當放大層級
m = folium.Map(location=center_coords, zoom_start=6)

# 2. 標示中心節點 (紅色)
folium.Marker(
    location=center_coords,
    popup=f"<b>中心點 ID: {center_node_id}</b>",
    tooltip=f"中心點: {center_node_id}",
    icon=folium.Icon(color="red", icon="star")
).add_to(m)

# 3. 取得相連的邊，並把目標節點跟線畫出來
connected_edges = my_geograph.graph[center_node_id]

for target_id, weight in connected_edges.items():
    # 動態從圖資中抓出目標節點的座標
    target_coords = my_geograph.nodes[target_id]
    
    # 畫出目標節點 (藍色)
    folium.Marker(
        location=target_coords,
        popup=f"節點 ID: {target_id}",
        tooltip=f"ID: {target_id}",
        icon=folium.Icon(color="blue", icon="info-sign")
    ).add_to(m)
    
    # 畫出兩點之間的連線
    folium.PolyLine(
        locations=[center_coords, target_coords],
        color="gray",
        weight=3,
        opacity=0.7,
        tooltip=f"權重: {weight}"
    ).add_to(m)
    
    # 計算線段的中點，用來放權重的文字標籤
    mid_lat = (center_coords[0] + target_coords[0]) / 2
    mid_lon = (center_coords[1] + target_coords[1]) / 2
    
    # 在中點加上 DivIcon，直接在圖面上顯示數值 (稍微四捨五入取到小數第一位比較乾淨)
    folium.Marker(
        location=[mid_lat, mid_lon],
        icon=folium.DivIcon(
            html=f'<div style="font-size: 10pt; color: black; font-weight: bold; background-color: rgba(255,255,255,0.8); padding: 2px 4px; border-radius: 4px; border: 1px solid black; white-space: nowrap; transform: translate(-50%, -50%);">{weight:.1f}</div>'
        )
    ).add_to(m)

# 4. 將地圖存成 HTML 檔案
output_file = "scgraph_node_visualization.html"
m.save(output_file)

# 5. 使用瀏覽器自動開啟該 HTML 檔案
# 取得絕對路徑，避免瀏覽器找不到檔案
absolute_path = os.path.abspath(output_file)
webbrowser.open(f"file://{absolute_path}")

print("在地圖已生成並嘗試於瀏覽器開啟！請確認畫面。")

在地圖已生成並嘗試於瀏覽器開啟！請確認畫面。


In [13]:
# 您要封閉的路段兩端點
node_A = 6872  # 剛剛找到的節點 (例如 105)
node_B = 6754              # 您看終端機印出後，決定要切斷的目標節點

penalty_weight = 9999999
# penalty_weight = 489.4

# 將 A 走到 B 的距離設為極大值
if node_B in my_geograph.graph[node_A]:
    my_geograph.graph[node_A][node_B] = penalty_weight
    print(f"已將 {node_A} -> {node_B} 的權重設為 {penalty_weight}")

# 將 B 走到 A 的距離設為極大值 (確保雙向封閉)
if node_A in my_geograph.graph[node_B]:
    my_geograph.graph[node_B][node_A] = penalty_weight
    print(f"已將 {node_B} -> {node_A} 的權重設為 {penalty_weight}")

已將 6872 -> 6754 的權重設為 9999999
已將 6754 -> 6872 的權重設為 9999999


## Run Scgraph to test

In [15]:
import folium
import webbrowser
import os

# ⚠️ 假設您已經在同一個環境中載入了 my_geograph，並且已經執行了「修改權重為 9999999」的程式碼

# ==========================================
# 1. 設定您想測試的起點與終點座標
# (建議挑選兩個「原本會經過節點 6872」，但現在應該要繞道的座標)
# ==========================================
origin_coords = {"latitude": 29.628277, "longitude": 122.283264}      # 替換成您的測試起點
destination_coords = {"latitude": -20.571120, "longitude": 117.195677} # 替換成您的測試終點
# (122.283264, 29.628277), (117.195677, -20.571120)
print("正在計算最短路徑，請稍候...")

# ==========================================
# 2. 呼叫 scgraph 內建的路徑規劃功能
# ==========================================
result = my_geograph.get_shortest_path(
    origin_node=origin_coords,
    destination_node=destination_coords
)

print(f" 計算完成！總航行距離: {result['length']:.2f}")

# 提取路徑的經緯度清單
# 為了相容 scgraph 不同版本的回傳格式 (list of dicts 或 list of lists)，這裡做個安全轉換
path_coordinates = []
for pt in result['coordinate_path']:
    if isinstance(pt, dict):
        path_coordinates.append([pt['latitude'], pt['longitude']])
    else:
        path_coordinates.append([pt[0], pt[1]])

# ==========================================
# 3. 將規劃出的路徑畫在 Folium 地圖上
# ==========================================
# 以起點為中心建立地圖
m_path = folium.Map(location=path_coordinates[0], zoom_start=6)

# 標示起點 (綠色)
folium.Marker(
    location=path_coordinates[0],
    popup="起點",
    icon=folium.Icon(color="green", icon="play")
).add_to(m_path)

# 標示終點 (紅色)
folium.Marker(
    location=path_coordinates[-1],
    popup="終點",
    icon=folium.Icon(color="red", icon="stop")
).add_to(m_path)

# 畫出導航路線 (藍色粗線)
folium.PolyLine(
    locations=path_coordinates,
    color="blue",
    weight=5,
    opacity=0.8,
    tooltip="scgraph 規劃路徑"
).add_to(m_path)

# 將地圖存成 HTML 並自動開啟
output_path_file = "scgraph_routed_path.html"
m_path.save(output_path_file)

absolute_path = os.path.abspath(output_path_file)
webbrowser.open(f"file://{absolute_path}")

print(" 導航路徑已生成並嘗試於瀏覽器開啟！請確認路線是否有成功避開您設定的高權重路段。")

正在計算最短路徑，請稍候...
 計算完成！總航行距離: 6178.83
 導航路徑已生成並嘗試於瀏覽器開啟！請確認路線是否有成功避開您設定的高權重路段。
